In [65]:
import time
import pandas as pd
import requests
import urllib.parse as urlparse
import datetime
import urllib3
import math
from dateutil import parser as dtparser
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

In [2]:
start = "41.134073194962816,16.8184268094641"               # San Francisco
end = "41.131452672190235,16.827025971567991"                # Los Angeles
key = "SdbkkAPVV6GxzS7beuYj8mqYnSRWgUmx"                 # API Key

# Base URL
base_url = "https://api.tomtom.com/routing/1/calculateRoute/"

In [66]:
route_summaries = pd.DataFrame()

today = datetime.date.today()
print(today)
departure_time_start = datetime.datetime(today.year, today.month, today.day - 1, 0, 0, 0)
hour_range = range(0, 48)

for i in hour_range:

    minute, hour = math.modf(i / 2)
    # Calculate hours and minutes shift
    hours = round(hour)
    minutes = round(minute * 60)
    
    # Update an hour
    departure_time = departure_time_start.replace(hour=departure_time_start.hour + hours, minute=departure_time_start.minute + minutes)
    
    # Format datetime string
    departure_time = departure_time.strftime('%Y-%m-%dT%H:%M:%S')

    # Create request URL
    request_params = (
        urlparse.quote(start) + ":" + urlparse.quote(end) 
        # c_start + ":" + c_end 
        + "/json?departAt=" + urlparse.quote(departure_time)
        + "&sectionType=traffic&report=effectiveSettings&traffic=true&travelMode=car&computeTravelTimeFor=all"
    )
    request_url = base_url + request_params + "&key=" + key

    # Get data
    response = requests.get(request_url)

    # Convert to JSON
    json_result = response.json()

    # Get summary
    route_summary = json_result['routes'][0]['summary']

    # Collect data in DataFrame
    route_summaries = pd.concat([route_summaries, pd.json_normalize(route_summary)], ignore_index=True)
        
    print(f"Retrieving data: {i+1} / {len(hour_range)} >> {departure_time} ... ({response.status_code})")

    time.sleep(0.3)



2024-07-29
Retrieving data: 1 / 48 >> 2024-07-28T00:00:00 ... (200)
Retrieving data: 2 / 48 >> 2024-07-28T00:30:00 ... (200)
Retrieving data: 3 / 48 >> 2024-07-28T01:00:00 ... (200)
Retrieving data: 4 / 48 >> 2024-07-28T01:30:00 ... (200)
Retrieving data: 5 / 48 >> 2024-07-28T02:00:00 ... (200)
Retrieving data: 6 / 48 >> 2024-07-28T02:30:00 ... (200)
Retrieving data: 7 / 48 >> 2024-07-28T03:00:00 ... (200)
Retrieving data: 8 / 48 >> 2024-07-28T03:30:00 ... (200)
Retrieving data: 9 / 48 >> 2024-07-28T04:00:00 ... (200)
Retrieving data: 10 / 48 >> 2024-07-28T04:30:00 ... (200)
Retrieving data: 11 / 48 >> 2024-07-28T05:00:00 ... (200)
Retrieving data: 12 / 48 >> 2024-07-28T05:30:00 ... (200)
Retrieving data: 13 / 48 >> 2024-07-28T06:00:00 ... (200)
Retrieving data: 14 / 48 >> 2024-07-28T06:30:00 ... (200)
Retrieving data: 15 / 48 >> 2024-07-28T07:00:00 ... (200)
Retrieving data: 16 / 48 >> 2024-07-28T07:30:00 ... (200)
Retrieving data: 17 / 48 >> 2024-07-28T08:00:00 ... (200)
Retrieving d

In [47]:
route_summaries

,lengthInMeters,travelTimeInSeconds,trafficDelayInSeconds,trafficLengthInMeters,departureTime,arrivalTime,noTrafficTravelTimeInSeconds,historicTrafficTravelTimeInSeconds,liveTrafficIncidentsTravelTimeInSeconds,time
0,778,60,0,0,2024-07-28T00:00:00+02:00,2024-07-28T00:00:59+02:00,60,60,60,2024-07-28T00:00:00
1,778,60,0,0,2024-07-28T00:30:00+02:00,2024-07-28T00:30:59+02:00,60,60,60,2024-07-28T00:30:00
2,778,60,0,0,2024-07-28T01:00:00+02:00,2024-07-28T01:00:59+02:00,60,60,60,2024-07-28T01:00:00
3,778,60,0,0,2024-07-28T01:30:00+02:00,2024-07-28T01:30:59+02:00,60,60,60,2024-07-28T01:30:00
4,778,60,0,0,2024-07-28T02:00:00+02:00,2024-07-28T02:00:59+02:00,60,60,60,2024-07-28T02:00:00
5,778,60,0,0,2024-07-28T02:30:00+02:00,2024-07-28T02:30:59+02:00,60,60,60,2024-07-28T02:30:00
6,778,60,0,0,2024-07-28T03:00:00+02:00,2024-07-28T03:00:59+02:00,60,60,60,2024-07-28T03:00:00
7,778,60,0,0,2024-07-28T03:30:00+02:00,2024-07-28T03:30:59+02:00,60,60,60,2024-07-28T03:30:00
8,778,60,0,0,2024-07-28T04:00:00+02:00,2024-07-28T04:01:00+02:00,60,60,60,2024-07-28T04:00:00
9,778,60,0,0,2024-07-28T04:30:00+02:00,2024-07-28T04:31:00+02:00,60,60,60,2024-07-28T04:30:00


In [59]:
route_analysis_df = pd.DataFrame()

for index, summary in route_summaries.iterrows():
    # Get length of road
    length_in_meters = summary['lengthInMeters']
    # Calculate free flow speed in km/h
    free_flow_speed = length_in_meters / summary['noTrafficTravelTimeInSeconds']  * 3.6
    # Calculate traffic speed in km/h
    traffic_speed = length_in_meters / summary['travelTimeInSeconds']  * 3.6
    # Calculate traffic density
    density_factor = free_flow_speed / traffic_speed
    avg_vehicle_length = 4.6
    
    # https://www.amsi.org.au/teacher_modules/pdfs/Maths_delivers/Braking5.pdf
    breaking_distance = (traffic_speed ** 2) / 20 # b = m/s 
    
    # The number of vehicles is calculated as the traffic density * length of road divided by the average length of vehicles and recommended breaking distance
    vehicles = round(density_factor * length_in_meters / avg_vehicle_length)

    # Departure time
    datetime = summary['departureTime']
    
    results = {"datetime": datetime, "free_flow_speed": free_flow_speed, "traffic_speed": traffic_speed, "density_factor": density_factor, "number_of_vehicles": vehicles }
    
    # Convert to data frame and append
    route_analysis_df = pd.concat([route_analysis_df, pd.json_normalize(results)], ignore_index=True)

dt = dtparser.parse(route_analysis_df.at[0, 'datetime'])
route_analysis_df.to_csv(f'{dt.year}-{dt.month:02d}-{dt.day:02d}.csv', index=False)

In [49]:
route_analysis_df

,time,free_flow_speed,traffic_speed,density_factor,number_of_vehicles
0,2024-07-28T00:00:00+02:00,46.68,46.680000,1.000000,169
1,2024-07-28T00:30:00+02:00,46.68,46.680000,1.000000,169
2,2024-07-28T01:00:00+02:00,46.68,46.680000,1.000000,169
3,2024-07-28T01:30:00+02:00,46.68,46.680000,1.000000,169
4,2024-07-28T02:00:00+02:00,46.68,46.680000,1.000000,169
5,2024-07-28T02:30:00+02:00,46.68,46.680000,1.000000,169
6,2024-07-28T03:00:00+02:00,46.68,46.680000,1.000000,169
7,2024-07-28T03:30:00+02:00,46.68,46.680000,1.000000,169
8,2024-07-28T04:00:00+02:00,46.68,46.680000,1.000000,169
9,2024-07-28T04:30:00+02:00,46.68,46.680000,1.000000,169
